# Study 821 — Turnover Volatility 🌀

**Do stocks with the most *erratic* trading go on to earn *less*?**

Chordia, Subrahmanyam & Anshuman (2001) find that beyond the *level* of trading
activity, its **variability** predicts the cross-section of returns *negatively*: the
names whose daily **turnover** is most unpredictable (a high **coefficient of variation**
of turnover) under-earn — a liquidity-risk discount. A long **low**-turnover-vol /
short **high**-turnover-vol book should earn a positive spread. We take the
self-contained daily version on a liquid US cross-section (2010-01-04 → 2026-06-30,
50 names).

*Numbers below are the frozen headline (`docs/results.md`); the live cells run the fast
synthetic control. Survivorship: current-membership mega-caps — magnitudes are an upper
bound.*


## 1. The idea in one picture

Turnover measures how briskly a stock changes hands. Two names can trade the *same* average volume, yet one does so like clockwork while the other lurches between frenzies and droughts. Chordia-Subrahmanyam-Anshuman argue that the **erratic** one is riskier to hold — you might need to sell exactly when its liquidity has evaporated — so it is priced at a discount and should *under*-earn. Sort on the trailing coefficient of variation of turnover; buy the steady names, sell the erratic ones.

In [1]:
import numpy as np, pandas as pd
R = dict(spread_bps=-1.7, t_nw=-1.73, lo_bps=5.95, hi_bps=7.65, gross_sharpe=-0.41)
print('long low-vol / short high-vol spread: %+.2f bps/day (NW t = %+.2f)'
      % (R['spread_bps'], R['t_nw']))
print('  low-turnover-vol book %+.2f bps vs high-turnover-vol book %+.2f bps'
      % (R['lo_bps'], R['hi_bps']))
print('  gross spread Sharpe (before cost): %.2f' % R['gross_sharpe'])

long low-vol / short high-vol spread: -1.70 bps/day (NW t = -1.73)
  low-turnover-vol book +5.95 bps vs high-turnover-vol book +7.65 bps
  gross spread Sharpe (before cost): -0.41


## 2. Is the sort just lucky? A live synthetic control

We plant the effect in a seeded toy world (`edge>0`: erratic turnover → lower forward return) and check the detector recovers it — and that it stays *silent* on the null (`edge=0`, turnover-vol present but unpriced). No network.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from turnover_vol import data, strategy as st
null = st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=821, n_assets=40, n_days=1200))
planted = st.synthetic_detect(data.synthetic_panel(edge=0.0016, seed=821, n_assets=40, n_days=1500))
print('null world   : spread NW t = %+.2f  (should be ~0)' % null['t_nw'])
print('planted world: spread NW t = %+.2f  (should light up)' % planted['t_nw'])

null world   : spread NW t = +1.13  (should be ~0)
planted world: spread NW t = +9.33  (should light up)


## 3. The honest verdict — the famous edge does *not* replicate here

On this liquid mega-cap tape the long-low-vol / short-high-vol spread is **-1.70 bps/day** with NW *t* = **-1.73** — statistically **insignificant** (|t| < 2), and if anything faintly the *wrong* sign versus the claim (the erratic names slightly *out*-earned). It is carried entirely by the pre-2018 era (*t* = -2.21) and gone thereafter (*t* = -0.36); the permutation null sits at zero and the observed value is only ~1.9σ from it. The seeded synthetic control recovers a *planted* CSA relation cleanly, so the machinery is sound — the turnover-variability premium is a small/illiquid-stock phenomenon that does not survive on 50 mega-caps. **Signal: None** (the claimed edge is absent), **Tradability: Mirage**.